# 스트레스 점수 예측 V8

**Multi-seed Quantile ExtraTrees**

- 행 단위 파생변수 생성
- Train 데이터만 이용해 전처리 기준 학습
- ExtraTrees 1,200개 × 3개 시드
- 각 모델의 51% 분위수 예측을 평균
- 최종 예측값을 0~1 범위로 제한

In [18]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

## 1. 설정

In [19]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
SEEDS = (42, 77, 2026)
N_ESTIMATORS = 1200
QUANTILE = 0.51

WEIGHTED_FEATURES = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

## 2. 데이터 불러오기 및 검증

In [20]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if list(submission.columns) != [ID_COLUMN, TARGET]:
    raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
if train[TARGET].isna().any():
    raise ValueError("Train 타깃에 결측값이 있습니다.")

print("Train:", train.shape)
print("Test:", test.shape)
train.head()

Train: (3000, 18)
Test: (3000, 17)


,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


## 3. 행 단위 파생변수

In [21]:
def add_features(frame):
    # 다른 행이나 데이터셋의 통계를 사용하지 않습니다.
    result = frame.copy()

    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)

    # max_features=1에서 중요 피처의 선택 확률을 높입니다.
    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]

    return result


X_train = add_features(train.drop(columns=[TARGET, ID_COLUMN]))
X_test = add_features(test.drop(columns=[ID_COLUMN]))
y_train = train[TARGET]

print("파생변수 적용 후:", X_train.shape)

파생변수 적용 후: (3000, 38)


## 4. 누수 없는 전처리

In [22]:
categorical = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
numerical = X_train.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    [
        (
            "categorical",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="constant",
                            fill_value="missing",
                        ),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(strategy="median", add_indicator=True),
            numerical,
        ),
    ]
)

# fit은 Train에만 수행하고 Test에는 transform만 수행합니다.
train_matrix = preprocessor.fit_transform(X_train)
test_matrix = preprocessor.transform(X_test)

print("전처리 완료:", train_matrix.shape, test_matrix.shape)

전처리 완료: (3000, 57) (3000, 57)


## 5. 5-Fold 교차검증

각 Fold의 Train 부분에서만 전처리 기준과 모델을 학습합니다.  
세 개 시드의 51% 분위수 예측을 평균한 뒤 MAE를 계산합니다.

In [23]:
def predict_forest_quantile(model, matrix, quantile=QUANTILE):
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )
    return np.quantile(tree_predictions, quantile, axis=1)


cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

fold_mae = []
oof_v8 = np.zeros(len(X_train))

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X_train),
    start=1,
):
    # Fold의 학습 데이터로만 전처리 기준을 학습합니다.
    fold_preprocessor = clone(preprocessor)
    fold_train_matrix = fold_preprocessor.fit_transform(
        X_train.iloc[train_idx]
    )
    fold_valid_matrix = fold_preprocessor.transform(
        X_train.iloc[valid_idx]
    )

    fold_seed_predictions = []

    for seed in SEEDS:
        fold_model = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=1,
            max_features=1,
            random_state=seed,
            n_jobs=-1,
        )
        fold_model.fit(
            fold_train_matrix,
            y_train.iloc[train_idx],
        )
        seed_prediction = predict_forest_quantile(
            fold_model,
            fold_valid_matrix,
        )
        fold_seed_predictions.append(seed_prediction)

    valid_prediction = np.mean(
        fold_seed_predictions,
        axis=0,
    )
    valid_prediction = np.clip(valid_prediction, 0.0, 1.0)
    oof_v8[valid_idx] = valid_prediction

    score = mean_absolute_error(
        y_train.iloc[valid_idx],
        valid_prediction,
    )
    fold_mae.append(score)
    print(f"Fold {fold} MAE: {score:.6f}")

fold_mae = np.array(fold_mae)
v8_cv_mae = mean_absolute_error(y_train, oof_v8)

print("V8 Fold MAE:", np.round(fold_mae, 6))
print("V8 Mean MAE:", f"{v8_cv_mae:.6f}")
print("V8 MAE Std:", f"{fold_mae.std():.6f}")

V6_CV_MAE = 0.149243
print("V6 Mean MAE:", f"{V6_CV_MAE:.6f}")
print("V8 vs V6:", f"{v8_cv_mae - V6_CV_MAE:+.6f}")

Fold 1 MAE: 0.135718
Fold 2 MAE: 0.151219
Fold 3 MAE: 0.148485
Fold 4 MAE: 0.158967
Fold 5 MAE: 0.149739
V8 Fold MAE: [0.135718 0.151219 0.148485 0.158967 0.149739]
V8 Mean MAE: 0.148826
V8 MAE Std: 0.007501
V6 Mean MAE: 0.149243
V8 vs V6: -0.000417


## 6. 전체 Train 학습 및 Test 예측

In [24]:
def predict_forest_quantile(model, matrix, quantile=QUANTILE):
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )
    return np.quantile(tree_predictions, quantile, axis=1)


seed_predictions = []

for seed in SEEDS:
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(train_matrix, y_train)
    prediction = predict_forest_quantile(model, test_matrix)
    seed_predictions.append(prediction)
    print(f"Seed {seed}: prediction complete")

Seed 42: prediction complete
Seed 77: prediction complete
Seed 2026: prediction complete


## 7. 최종 예측값 확인

In [25]:
pred_v8 = np.mean(seed_predictions, axis=0)
pred_v8 = np.clip(pred_v8, 0.0, 1.0)

print("Prediction shape:", pred_v8.shape)
print("Prediction range:", pred_v8.min(), pred_v8.max())
print("Prediction mean:", pred_v8.mean())
print("첫 10개 예측값:")
print(np.round(pred_v8[:10], 6))


Prediction shape: (3000,)
Prediction range: 0.0 1.0
Prediction mean: 0.5023654333333333
첫 10개 예측값:
[0.581633 0.97     0.19     0.45     0.53     0.46     0.78     0.48
 0.536667 0.88    ]


## 8. 제출 파일 생성 및 확인

In [26]:
submission_v8 = submission.copy()
submission_v8[ID_COLUMN] = test[ID_COLUMN]
submission_v8[TARGET] = pred_v8

output_path = OUTPUT_DIR / "submit_v8_multiseed_quantile_forest.csv"
submission_v8.to_csv(output_path, index=False, encoding="utf-8")

print("Saved:", output_path)
print("Rows:", len(submission_v8))
submission_v8.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v8_multiseed_quantile_forest.csv
Rows: 3000


,ID,stress_score
0,TEST_0000,0.581633
1,TEST_0001,0.970000
2,TEST_0002,0.190000
3,TEST_0003,0.450000
4,TEST_0004,0.530000
5,TEST_0005,0.460000
6,TEST_0006,0.780000
7,TEST_0007,0.480000
8,TEST_0008,0.536667
9,TEST_0009,0.880000


### 예측값 요약 통계

In [27]:
submission_v8[TARGET].describe().to_frame(name="stress_score")

,stress_score
count,3000.000000
mean,0.502365
std,0.202849
min,0.000000
25%,0.420000
50%,0.506667
75%,0.580000
max,1.000000
